In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

## Input: Limit Order Book and OHLCV data with volatility for Agent Training

In [2]:
df = pd.read_csv("merged_bid_ask_ohlcv_data.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)


In [3]:
# Effective Bid Ask Spread
def EffectiveBidAskSpread(data, window_size=60):
    ''' Spread is to measure liquidity, larger spread, less liquidity
    '''
    result = data[['timestamp','close']].copy()
    result['Delta_P_t'] = result['close'].diff()
    result['Delta_P_shift'] = result['Delta_P_t'].shift()
    
    result['EffectiveBidAskSpread'] = (
        result['Delta_P_t'].
                   rolling(window=window_size).cov(result['Delta_P_shift']).
                   apply(lambda x: max(0,(-x))**0.5)
    )
    data['EffectiveSpread'] = result['EffectiveBidAskSpread']
    
    return data
df = EffectiveBidAskSpread(df) # add Effective Spread

In [4]:
# High Low Volatility
def HLVolatility1(data, window_size=60):
    result = pd.DataFrame()
    result['timestamp'] = data['timestamp']
    
    result['HL_Volatility'] = (np.log(data['high'])-np.log(data['low']))**2
    result['HL_Volatility'] = result['HL_Volatility'].rolling(window=window_size).mean()
    result['HL_Volatility'] = (result['HL_Volatility']/(4*np.log(2)))**0.5
    df['HLVolatility'] = result['HL_Volatility']
    
    return df
df = HLVolatility1(df)

In [5]:
# Corwin-Schultz Spread
def CSSpread(data, window_size=60):
    result = pd.DataFrame()
    result['timestamp'] = data['timestamp']
    result['gama'] = (np.log(np.maximum(data['high'], data['high'].shift(1))/np.minimum(data['high'], data['high'].shift(1))))**2
    
    result['beta'] = (np.log(data['high']/data['low']))**2
    result['beta'] = (result['beta']+result['beta'].shift(1)).rolling(window=window_size).mean()
    
    result['alpha'] = result['beta']**0.5 * (2**0.5-1) / (3-2*2**0.5) - (result['gama'] / (3-2*2**0.5))**0.5
    result['alpha'] = result['alpha'].apply(lambda x:max(x,0)) # avoid negative spread
    
    result['S'] = 2*(np.exp(result['alpha'])-1)/(np.exp(result['alpha'])+1)
    df['CS_Spread'] = result['S']
    
    return df
df = CSSpread(df)

In [6]:
import statsmodels.api as sm
# Kyle's Lambda -> 1/lambda implies liquidity
def Kyle_Lambda(data):
    df = pd.DataFrame()
    df['date'] = data['timestamp'].dt.date
    df['delta_p_t'] = data['close'] - data['close'].shift(1)
    df['delta_price'] = data['close'] - data['close'].shift(1)
    df['b_t'] = np.sign(df['delta_price'])
    df['b_t'] = df['b_t'].replace(0, method='ffill')
    df['b_t'] = df['b_t'].fillna(1)
    df['signed_volume'] = df['b_t'] * data['volume']
    kyle_lambda_list = []
    grouped = df.dropna(subset=['delta_p_t', 'signed_volume']).groupby('date')
    for _, group in grouped:
        if len(group) > 1:
            X, y = group['signed_volume'], group['delta_p_t']
            X = sm.add_constant(X)
            model = sm.OLS(y, X)
            results = model.fit()
            lambda_hat = results.params['signed_volume']
            df.loc[group.index, 'Kyle_Lambda'] = lambda_hat
        else:
            df.loc[group.index, 'Kyle_Lambda'] = np.nan
    data['Kyle_Lambda'] = df['Kyle_Lambda']
    return data
df = Kyle_Lambda(df)

/var/folders/73/r6fj47bs1vz8m5q3xtt377l80000gn/T/ipykernel_57089/1801812791.py:9: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df['b_t'] = df['b_t'].replace(0, method='ffill')


In [7]:
# Volume-Synchronized Probability of Informed Trading (VPIN) -> lead volatility
def VolumePIN(data, window_size=60):
    df = pd.DataFrame()
    df['Total_Bid_Size'] = data[['bid_size_1', 'bid_size_2', 'bid_size_3', 'bid_size_4', 'bid_size_5']].sum(axis=1)
    df['Total_Ask_Size'] = data[['ask_size_1', 'ask_size_2', 'ask_size_3', 'ask_size_4', 'ask_size_5']].sum(axis=1)
    df['OBI'] = (df['Total_Bid_Size'] - df['Total_Ask_Size']) / (df['Total_Bid_Size'] + df['Total_Ask_Size'])
    df['OBI'] = df['OBI'].replace([np.inf, -np.inf], np.nan).fillna(0)
    df['V_B'] = data['volume'] * (1 + df['OBI']) / 2 # estimate V^B
    df['V_S'] = data['volume'] * (1 - df['OBI']) / 2 # estimate V^S
    df['sum_V_B'] = df['V_B'].rolling(window=window_size).sum()
    df['sum_V_S'] = df['V_S'].rolling(window=window_size).sum()
    df['V'] = data['volume'].rolling(window=window_size).mean()
    df['numerator'] = (df['V_B'] - df['V_S']).abs().rolling(window=window_size).sum()
    df['denominator'] = window_size * df['V']
    df['denominator'] = df['denominator'].replace(0, np.nan)
    data['Volume_PIN'] = df['numerator'] / df['denominator']
    return data
df = VolumePIN(df)

In [8]:
# Fibonacci Retracement (maybe not so useful) -> support and resistance
def Fibo_Retrace(data, window_size=390):
    period_high = data['high'].rolling(window=window_size, min_periods=1).max()
    period_low = data['low'].rolling(window=window_size, min_periods=1).min()
    for i,level in enumerate([0.236, 0.382, 0.5, 0.618, 0.786]):
        data[f'Fib_Level_{i}'] = period_high - (period_high - period_low) * level
    return data
df = Fibo_Retrace(df)

## ADD Sentiment: check here:
## The file 'news_sentiment_to_merge.csv' is generated by Features_News_sentiment.ipynb
## Ticker = 'AAPL' and use Free Alpha Vantage API to fetch news

In [9]:
# Big event indicators as indicator function 0 or 1
# def big_event_indicator(stock_data, event_dates):
#     """
#     Add a binary indicator (0 or 1) for the occurrence of a big event.

#     Parameters:
#     - stock_data (pd.DataFrame): The stock data.
#     - event_dates (list of str): List of event dates in 'YYYY-MM-DD' format.
#     """
#     big_event_indicator = np.zeros(len(stock_data))
#     for event_date in event_dates:
#         event_date = pd.Timestamp(event_date)  # Convert to pd.Timestamp
#         big_event_indicator[(stock_data.index >= event_date) & (stock_data.index < event_date + pd.Timedelta(hours=1))] = 1
#     stock_data['Big Event Indicator'] = pd.Series(big_event_indicator, index=stock_data.index)
#     return stock_data
sentiment = pd.read_csv('news_sentiment_to_merge.csv')
sentiment = sentiment.rename(columns={"time_published": "timestamp"})
sentiment['timestamp'] = pd.to_datetime(sentiment['timestamp'], utc=True)
sentiment=sentiment.drop_duplicates(subset='timestamp', keep='last')
df = pd.merge(df, sentiment, on='timestamp', how='left')
df['news_sentiment_score'] = df['news_sentiment_score'].fillna(0)

## sentiment is a sparse signal

In [10]:
len(df[df['news_sentiment_score']>0])/len(df['news_sentiment_score'])

0.0020799492888554337

In [11]:
df.iloc[100:110,-10:]

,HLVolatility,CS_Spread,Kyle_Lambda,Volume_PIN,Fib_Level_0,Fib_Level_1,Fib_Level_2,Fib_Level_3,Fib_Level_4,news_sentiment_score
100,0.000645,0.001901,0.000001,0.286293,178.74356,178.26322,177.875,177.48678,176.93406,0.0
101,0.000645,0.002708,0.000001,0.286614,178.74356,178.26322,177.875,177.48678,176.93406,0.0
102,0.000644,0.003664,0.000001,0.276853,178.74356,178.26322,177.875,177.48678,176.93406,0.0
103,0.000645,0.003392,0.000001,0.278046,178.74356,178.26322,177.875,177.48678,176.93406,0.0
104,0.000643,0.003663,0.000001,0.283555,178.74356,178.26322,177.875,177.48678,176.93406,0.0
105,0.000641,0.003651,0.000001,0.283426,178.74356,178.26322,177.875,177.48678,176.93406,0.0
106,0.000639,0.002406,0.000001,0.284223,178.74356,178.26322,177.875,177.48678,176.93406,0.0
107,0.000638,0.003355,0.000001,0.283809,178.74356,178.26322,177.875,177.48678,176.93406,0.0
108,0.000636,0.003072,0.000001,0.281541,178.74356,178.26322,177.875,177.48678,176.93406,0.0
109,0.000635,0.003612,0.000001,0.281605,178.74356,178.26322,177.875,177.48678,176.93406,0.0


## Test colinearity / PCA
## 1. volatility group: volatility / HLVolatility / news_sentiment
## 2. liquidity: spread / lambda
## 3. alpha: Volume_PIN / 
## 4. LOB / OLHCV / 

In [13]:
df.columns

Index(['timestamp', 'bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4',
       'bid_price_5', 'bid_size_1', 'bid_size_2', 'bid_size_3', 'bid_size_4',
       'bid_size_5', 'ask_price_1', 'ask_price_2', 'ask_price_3',
       'ask_price_4', 'ask_price_5', 'ask_size_1', 'ask_size_2', 'ask_size_3',
       'ask_size_4', 'ask_size_5', 'rtype', 'publisher_id', 'instrument_id',
       'open', 'high', 'low', 'close', 'volume', 'symbol', 'log_return',
       'volatility', 'EffectiveSpread', 'HLVolatility', 'CS_Spread',
       'Kyle_Lambda', 'Volume_PIN', 'Fib_Level_0', 'Fib_Level_1',
       'Fib_Level_2', 'Fib_Level_3', 'Fib_Level_4', 'news_sentiment_score'],
      dtype='object')

In [19]:
df[['CS_Spread','EffectiveSpread','Kyle_Lambda']].corr()

,CS_Spread,EffectiveSpread,Kyle_Lambda
CS_Spread,1.000000,0.391100,0.268476
EffectiveSpread,0.391100,1.000000,0.198068
Kyle_Lambda,0.268476,0.198068,1.000000
